# 0. Data Preprocessing & Feature Engineering Playground

In this notebook, we transform the raw Online Retail II dataset into a robust format suitable for Machine Learning models.

To keep our codebase scalable and easily maintainable, the lengthy blocks of data manipulation code have been refactored into modular python tools located in the `src/tools/` directory. This notebook now serves as an **interactive playground** mapping out the logical progression of our pipeline, allowing us to test each specific step in isolation before running the headless orchestration script `process_data.py`.

### Pipeline Overview:
1. **Imports & Loading**: Reading the raw Excel file into pandas via our centralized loader.
2. **Cleaning & Separation**: Dropping noise and splitting actual Sales from Returns.
3. **Weekly Aggregation**: Building the core temporal panel with zero-filling.
4. **Feature Engineering**: Injecting Calendar, Lags, and Pricing variables.
5. **Semantic Embeddings**: Calling Gemini API to understand product descriptions.
6. **Train/Test Profiling**: Slicing the timeline to prevent Data Leakage.
7. **Clustering**: Finding natural Behavioral and Volume clusters using HDBSCAN and Jenks Breaks.
8. **Final Joins**: Merging the ecosystem into the ML-ready Parquet format.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Ensure we can seamlessly import all modular functions from the src directory
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import our specialized pipeline tools
from src.tools.data_loader import load_raw_data
from src.tools.cleaning import split_sales_returns
from src.tools.add_temporal_features import add_temporal_features
from src.tools.feature_engineering import aggregate_weekly_sku, add_historical_features, add_pricing_features
from src.tools.clustering import calculate_demand_profile, calculate_commercial_profile, create_profile_clusters, create_volume_clusters
from src.tools.embeddings import embed_sku_descriptions

print("Setup complete. Environment mapped and modular tools loaded.")

Setup complete. Environment mapped and modular tools loaded.


## 1. Data Loading Tracker

Raw system exports are inherently messy. Before doing any mathematical operations, we need a baseline.
Our `load_raw_data` tool handles the heavy lifting by reading the Excel sheets and coercing datatypes.

In [8]:
print("Loading raw retail data (this might take a minute due to Excel format)...")
input_path = os.path.join(PROJECT_ROOT, "data/raw", "online_retail_II.xlsx")
raw_df = load_raw_data(input_path)

print(f"Data loaded. Shape: {raw_df.shape}")

raw_df

Loading raw retail data (this might take a minute due to Excel format)...
Data loaded. Shape: (1067371, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010
...,...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,Year 2010-2011
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,Year 2010-2011
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,Year 2010-2011
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France,Year 2010-2011


## 2. Cleaning & Sales/Returns Separation

We separate valid sales (strictly positive quantities, valid SKUs) from returns/cancellations. This prevents the target variable from dipping below zero, which would break many ML algorithms.

In [10]:
print("Separating valid sales from returns and cleaning text...")
sales_df, returns_df = split_sales_returns(raw_df)

print(f"Valid Sales Rows: {len(sales_df):,}")
print(f"Returns Rows: {len(returns_df):,}")

sales_df

Separating valid sales from returns and cleaning text...
Valid Sales Rows: 1,035,620
Returns Rows: 18,176


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,Week,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010,2009-12-01,83.40
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010,2009-12-01,81.00
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010,2009-12-01,81.00
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010,2009-12-01,100.80
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010,2009-12-01,30.00
...,...,...,...,...,...,...,...,...,...,...,...
1067365,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France,Year 2010-2011,2011-12-06,10.20
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,Year 2010-2011,2011-12-06,12.60
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,Year 2010-2011,2011-12-06,16.60
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,Year 2010-2011,2011-12-06,16.60


In [11]:
returns_df

,StockCode,Week,Quantity,Customer ID,Country
178,22087,2009-12-01,12,16321.0,Australia
179,85206A,2009-12-01,6,16321.0,Australia
180,21895,2009-12-01,4,16321.0,Australia
181,21896,2009-12-01,6,16321.0,Australia
182,22083,2009-12-01,12,16321.0,Australia
...,...,...,...,...,...
1065909,22178,2011-12-06,12,14397.0,United Kingdom
1065910,23144,2011-12-06,11,14397.0,United Kingdom
1067176,21258,2011-12-06,5,15311.0,United Kingdom
1067177,84978,2011-12-06,1,17315.0,United Kingdom


## 3. Weekly Aggregation & Zero-Filling

We transition from an event-driven transactional ledger to a strict weekly timeline. For forecasting to work, if a product didn't sell in Week 3, we must explicitly inject a `0` rather than skipping the week.

In [12]:
print("Aggregating transactions into weekly buckets per SKU (with continuous zero-filling)...")
weekly_sales = aggregate_weekly_sku(sales_df)

print(f"Weekly Panel Rows: {len(weekly_sales):,}")
display(weekly_sales.head(3))

Aggregating transactions into weekly buckets per SKU (with continuous zero-filling)...
Weekly Panel Rows: 511,665


,StockCode,Week,Quantity,Revenue
0,10002,2009-12-07,0,0.0
1,10002,2009-12-14,0,0.0
2,10002,2009-12-21,0,0.0


## 4. Advanced Feature Engineering

We inject three layers of ML features:
1. **Temporal**: Calendar variables and UK Holidays.
2. **Historical**: Lags (history) and rolling return rates (anomaly detection).
3. **Pricing**: Median static price and active promotional flags.

In [ ]:
print("Adding temporal (calendar) features...")
weekly_sales = add_temporal_features(weekly_sales)

print("Adding historical (lags & rolling return rates) features...")
weekly_sales = add_historical_features(weekly_sales, returns_df)

print("Adding pricing metrics (median price & promotional flags)...")
weekly_sales = add_pricing_features(weekly_sales, sales_df)

print("Feature Engineering complete!")
display(weekly_sales.head(3))

## 5. Semantic Embeddings

We use the Google Gemini API to translate textual product descriptions into 768-dimensional mathematical vectors. This allows algorithms to know that a "RED MUG" is similar to a "BLUE MUG". 
*Note: Results are cached as parquet to prevent redundant API billing.*

In [ ]:
print("Generating or loading semantic embeddings from product descriptions...")
embeddings_cache_path = os.path.join(PROJECT_ROOT, "Datasets", "embeddings_cache.parquet")
embeddings_df = embed_sku_descriptions(sales_df, cache_path=embeddings_cache_path)

print(f"Total Unique Products Embedded: {len(embeddings_df):,}")
display(embeddings_df[['StockCode', 'desc_canonical', 'embedding']].head(3))

## 6. Train/Test Split (Preventing Data Leakage)

If we calculate product clusters (total volume, inter-purchase intervals) across the entire dataset, we accidentally feed information from 2011 into models trained in 2010. We isolate the training data strictly before computing static clusters.

In [ ]:
print("Splitting data to calculate clusters using ONLY training data...")
# The dataset ends in Dec 2011. We use the last ~3 months for the test set.
TEST_CUTOFF = pd.to_datetime("2011-09-01")

weekly_sales_train = weekly_sales[weekly_sales["Week"] < TEST_CUTOFF]
sales_df_train = sales_df[sales_df["InvoiceDate"] < TEST_CUTOFF]

print(f"Training Rows Cutoff Complete.")

## 7. Dual Clustering Architecture

We build two sets of segmentations:
1. **Behavioral Profile**: HDBSCAN clustering on Demand (ADI/CV2) + Commercial + Semantic embeddings.
2. **Volume Tiers**: Using Jenks Natural Breaks to find the optimal skew thresholds for High/Medium/Low sellers.

In [ ]:
print("Building static SKU profiles (Demand & Commercial) on Train set...")
demand_df = calculate_demand_profile(weekly_sales_train)
commercial_df = calculate_commercial_profile(sales_df_train)

print("Creating Behavioral Clusters (Semantics + Demand + Commercial)...")
profile_clusters = create_profile_clusters(embeddings_df, demand_df, commercial_df)

print("Creating Volume Clusters (Jenks Natural Breaks) on Train set...")
volume_clusters = create_volume_clusters(weekly_sales_train, n_tiers=3)

display(profile_clusters.head(3))
display(volume_clusters.head(3))

## 8. Final Export

We merge the static clusters computed safely on the train set back onto the entire timeline (train + test), drop unclusterable anomalies, and export to parquet.

In [ ]:
print("Joining clusters back to the main weekly panel (Train + Test)...")
final_df = weekly_sales.merge(profile_clusters, on="StockCode", how="left")
final_df = final_df.merge(volume_clusters[["StockCode", "volume_cluster_id", "volume_tier"]], on="StockCode", how="left")

# Drop rows that didn't get clustered (e.g., products that only appeared in the test set or lacked text descriptions)
final_df = final_df.dropna(subset=["profile_cluster_id", "volume_cluster_id"])

output_path = os.path.join(PROJECT_ROOT, "Datasets", "processed_retail_data.parquet")
print(f"Exporting fully featured dataset to {output_path}...")
final_df.to_parquet(output_path, index=False)

print("Done! Ready for Machine Learning Models. 🚀")